# Checkpoint Upload / Download

Notebook này lấy `best` và `last` checkpoint từ một hoặc nhiều `run_name`, rồi upload lên Kaggle.
Script `download.sh` ở thư mục gốc dùng để tải checkpoint về local hoặc từ Kaggle dataset.

In [2]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "saved_results").exists() and (PROJECT_ROOT.parent / "saved_results").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

run_names = ['custom-baseline', 'torchvision-augmentmax', 'retina-baseline']

RUN_NAMES = list(dict.fromkeys(run_names))
TARGET = os.getenv("TARGET", "kaggle").strip().lower()
SAVED_RESULTS_ROOT = Path(os.getenv("SAVED_RESULTS_ROOT", PROJECT_ROOT / "saved_results"))
EXPORT_DIR = Path(os.getenv("EXPORT_DIR", SAVED_RESULTS_ROOT / "checkpoint_export_multi"))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET_SLUG = os.getenv("KAGGLE_DATASET_SLUG", "ngocbaotrinhtuan/object-detection").strip()
KAGGLE_DATASET_TITLE = os.getenv("KAGGLE_DATASET_TITLE", f"{len(RUN_NAMES)} runs checkpoints")
KAGGLE_LICENSE = os.getenv("KAGGLE_LICENSE", "CC0-1.0")

print({
    "project_root": str(PROJECT_ROOT),
    "run_names": RUN_NAMES,
    "target": TARGET,
    "saved_results_root": str(SAVED_RESULTS_ROOT),
    "export_dir": str(EXPORT_DIR),
})

{'project_root': '/home/uet/edu_viettel/xla/IM-object-detection', 'run_names': ['custom-baseline', 'torchvision-augmentmax', 'retina-baseline'], 'target': 'kaggle', 'saved_results_root': '/home/uet/edu_viettel/xla/IM-object-detection/saved_results', 'export_dir': '/home/uet/edu_viettel/xla/IM-object-detection/saved_results/checkpoint_export_multi'}


In [4]:
from getpass import getpass
from pathlib import Path
import json
import os

kaggle_username = os.getenv("KAGGLE_USERNAME", "ngocbao-trinhtuan").strip() or input("Kaggle username: ").strip()

In [ ]:
def latest_checkpoint(checkpoint_dir: Path, pattern: str, fallback_name: str) -> Path | None:
    matches = sorted(checkpoint_dir.glob(pattern), key=lambda path: path.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    fallback = checkpoint_dir / fallback_name
    return fallback if fallback.exists() else None

def stage_checkpoint(source: Path | None, destination: Path) -> Path | None:
    if source is None:
        return None
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return destination

def write_kaggle_metadata(export_dir: Path, dataset_slug: str, title: str, license_name: str) -> Path:
    metadata = {
        "title": title,
        "id": dataset_slug,
        "licenses": [{"name": license_name}],
    }
    metadata_path = export_dir / "dataset-metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return metadata_path

staged_runs = []
for run_name in RUN_NAMES:
    checkpoint_dir = SAVED_RESULTS_ROOT / run_name / "checkpoints"
    best_source = latest_checkpoint(checkpoint_dir, "best_model-*.pth", "best_model.pth")
    last_source = latest_checkpoint(checkpoint_dir, "last_model-*.pth", "last_model.pth")

    run_export_dir = EXPORT_DIR / run_name
    best_target = stage_checkpoint(best_source, run_export_dir / "best_model.pth")
    last_target = stage_checkpoint(last_source, run_export_dir / "last_model.pth")

    staged_runs.append(
        {
            "run_name": run_name,
            "checkpoint_dir": str(checkpoint_dir),
            "best_source": str(best_source) if best_source else None,
            "last_source": str(last_source) if last_source else None,
            "best_target": str(best_target) if best_target else None,
            "last_target": str(last_target) if last_target else None,
        }
    )

print(json.dumps(staged_runs, indent=2, ensure_ascii=False))

if not any(item["best_target"] or item["last_target"] for item in staged_runs):
    raise FileNotFoundError(f"No checkpoint found for all runs in {SAVED_RESULTS_ROOT}")

Wrote manifest: /home/uet/edu_viettel/xla/IM-object-detection/saved_results/checkpoint_export_multi/manifest.json
[
  {
    "run_name": "custom-baseline",
    "checkpoint_dir": "/home/uet/edu_viettel/xla/IM-object-detection/saved_results/custom-baseline/checkpoints",
    "best_source": "/home/uet/edu_viettel/xla/IM-object-detection/saved_results/custom-baseline/checkpoints/best_model-20260607-205358.pth",
    "last_source": "/home/uet/edu_viettel/xla/IM-object-detection/saved_results/custom-baseline/checkpoints/last_model-20260608-015352.pth",
    "best_target": "/home/uet/edu_viettel/xla/IM-object-detection/saved_results/checkpoint_export_multi/custom-baseline/best_model.pth",
    "last_target": "/home/uet/edu_viettel/xla/IM-object-detection/saved_results/checkpoint_export_multi/custom-baseline/last_model.pth"
  },
  {
    "run_name": "torchvision-augmentmax",
    "checkpoint_dir": "/home/uet/edu_viettel/xla/IM-object-detection/saved_results/torchvision-augmentmax/checkpoints",
    "b

In [7]:
if TARGET != "kaggle":
    raise ValueError("TARGET only supports 'kaggle' now.")

if not KAGGLE_DATASET_SLUG:
    raise ValueError("Set KAGGLE_DATASET_SLUG='owner/dataset-name' before uploading to Kaggle.")

metadata_path = write_kaggle_metadata(EXPORT_DIR, KAGGLE_DATASET_SLUG, KAGGLE_DATASET_TITLE, KAGGLE_LICENSE)
print(f"Kaggle metadata written: {metadata_path}")

command_version = [
    "kaggle",
    "datasets",
    "version",
    "-p",
    str(EXPORT_DIR),
    "--dir-mode",
    "zip",
    "-m",
    f"Update checkpoints for {len(RUN_NAMES)} runs: {', '.join(RUN_NAMES)}",
]
command_create = [
    "kaggle",
    "datasets",
    "create",
    "-p",
    str(EXPORT_DIR),
    "--dir-mode",
    "zip",
]

try:
    print("Uploading with Kaggle version...")
    subprocess.run(command_version, check=True)
except subprocess.CalledProcessError:
    print("Version upload failed, trying create...")
    subprocess.run(command_create, check=True)

Kaggle metadata written: /home/uet/edu_viettel/xla/IM-object-detection/saved_results/checkpoint_export_multi/dataset-metadata.json
Uploading with Kaggle version...
Starting upload for file custom-baseline.zip


100%|██████████| 712M/712M [03:14<00:00, 3.83MB/s]  


Upload successful: custom-baseline.zip (712MB)
Starting upload for file retina-baseline.zip


100%|██████████| 439M/439M [03:03<00:00, 2.51MB/s]   


Upload successful: retina-baseline.zip (439MB)
Starting upload for file torchvision-augmentmax.zip


100%|██████████| 845M/845M [03:07<00:00, 4.72MB/s]  


Upload successful: torchvision-augmentmax.zip (845MB)
Starting upload for file manifest.json


100%|██████████| 2.13k/2.13k [00:01<00:00, 2.02kB/s]


Upload successful: manifest.json (2KB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/ngocbaotrinhtuan/object-detection


## Download về local

Dùng script `download.sh` ở thư mục gốc:
- `SOURCE=local RUN_NAME=<run_name>` để copy trực tiếp từ `saved_results/<run_name>/checkpoints`
- `SOURCE=kaggle` để tải từ Kaggle dataset

Upload nhiều run trong notebook:
- Đặt `RUN_NAMES` qua env theo dạng CSV: `run-a,run-b,run-c`
- Hoặc JSON list: `["run-a", "run-b", "run-c"]`